# Notebook 2: Data Cleaning & Validation

## Objective

Validate and clean the master dark store dataset.

### Tasks

- Missing values
- Duplicate records
- Coordinate validation
- Brand validation
- Data quality checks

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
from google.colab import files

uploaded = files.upload()

Saving master_darkstores.csv to master_darkstores.csv


In [ ]:
stores = pd.read_csv("master_darkstores.csv")

In [ ]:
# Dataset Shape

print(f"Rows    : {stores.shape[0]}")
print(f"Columns : {stores.shape[1]}")

Rows    : 4081
Columns : 9


In [ ]:
# Dataset Information

stores.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4081 entries, 0 to 4080
Data columns (total 9 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   id         4081 non-null   object 
 1   accuracy   1954 non-null   float64
 2   latitude   4081 non-null   float64
 3   longitude  4081 non-null   float64
 4   brand      4081 non-null   object 
 5   name       1089 non-null   object 
 6   city       1089 non-null   object 
 7   state      1089 non-null   object 
 8   locality   1038 non-null   object 
dtypes: float64(3), object(6)
memory usage: 287.1+ KB


In [ ]:
# Preview Dataset

stores.head()

,id,accuracy,latitude,longitude,brand,name,city,state,locality
0,42985,43.0,8.478260,76.954711,Blinkit,NaN,NaN,NaN,NaN
1,40038,13.0,8.525199,76.955395,Blinkit,NaN,NaN,NaN,NaN
2,38514,30.0,9.587031,76.535782,Blinkit,NaN,NaN,NaN,NaN
3,38406,48.0,9.922391,78.095686,Blinkit,NaN,NaN,NaN,NaN
4,39056,0.0,9.949993,76.253442,Blinkit,NaN,NaN,NaN,NaN


In [ ]:
# Missing Value Percentage

missing_percentage = (
    stores.isnull()
    .mean()
    .mul(100)
    .round(2)
    .sort_values(ascending=False)
)

missing_percentage

,0
locality,74.57
name,73.32
state,73.32
city,73.32
accuracy,52.12
id,0.00
brand,0.00
latitude,0.00
longitude,0.00


In [ ]:
# Check Duplicate Rows

duplicate_rows = stores.duplicated().sum()

print("Duplicate Rows :", duplicate_rows)

Duplicate Rows : 0


In [ ]:
# Check Duplicate Coordinates

duplicate_coordinates = stores.duplicated(
    subset=["latitude", "longitude", "brand"]
).sum()

print("Duplicate Coordinates :", duplicate_coordinates)

Duplicate Coordinates : 0


In [ ]:
# Coordinate Summary

stores[["latitude", "longitude"]].describe()

,latitude,longitude
count,4081.000000,4081.000000
mean,21.494433,77.842709
std,6.285832,3.901752
min,8.184644,69.642790
25%,17.355889,75.829815
50%,21.276136,77.384920
75%,28.410694,78.572350
max,32.766852,94.932671


In [ ]:
# Check Invalid Latitude

stores[
    (stores["latitude"] < 6) |
    (stores["latitude"] > 38)
]

,id,accuracy,latitude,longitude,brand,name,city,state,locality


In [ ]:
# Check Invalid Longitude

stores[
    (stores["longitude"] < 68) |
    (stores["longitude"] > 98)
]

,id,accuracy,latitude,longitude,brand,name,city,state,locality


In [ ]:
# Brand Distribution

stores["brand"].value_counts()

,count
brand,
Blinkit,1954
Zepto,1089
Swiggy Instamart,1038


In [ ]:
# Brand Percentage

(
    stores["brand"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

,proportion
brand,
Blinkit,47.88
Zepto,26.68
Swiggy Instamart,25.43


In [ ]:
# Remove Duplicate Rows

stores = stores.drop_duplicates()

In [ ]:
# Reset Index

stores.reset_index(
    drop=True,
    inplace=True
)

In [ ]:
# Save Clean Dataset

stores.to_csv(
    "clean_darkstores.csv",
    index=False
)

print("✅ clean_darkstores.csv saved successfully.")

✅ clean_darkstores.csv saved successfully.


In [ ]:
# Download Clean Dataset

from google.colab import files

files.download("clean_darkstores.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import pandas as pd
import plotly.graph_objects as go

df = stores.copy()

# -----------------------------
# QUALITY GATES
# -----------------------------

total = len(df)

coordinates = (
    df["latitude"].notna() &
    df["longitude"].notna()
)

india_range = (
    coordinates &
    df["latitude"].between(6, 38) &
    df["longitude"].between(68, 98)
)

accuracy_available = (
    india_range &
    df["accuracy"].notna()
)

analysis_ready = accuracy_available


# -----------------------------
# FUNNEL DATA
# -----------------------------

stages = [
    "Raw Store Records",
    "Coordinates Available",
    "Within Geographic Range",
    "Accuracy Available",
    "Analysis Ready"
]

values = [
    total,
    coordinates.sum(),
    india_range.sum(),
    accuracy_available.sum(),
    analysis_ready.sum()
]


# -----------------------------
# FIGURE
# -----------------------------

fig = go.Figure(
    go.Funnel(
        y=stages,
        x=values,

        textinfo="value+percent initial",

        marker=dict(
            color=[
                "#202124",
                "#444444",
                "#777777",
                "#AAAAAA",
                "#2A9D8F"
            ]
        ),

        connector=dict(
            line=dict(
                color="#D9D9D9",
                width=2
            )
        ),

        hovertemplate=
        "<b>%{y}</b><br>" +
        "Records: %{x:,}<br>" +
        "Retention: %{percentInitial:.1%}" +
        "<extra></extra>"
    )
)

fig.update_layout(

    title=dict(
        text="<b>FROM RAW POINTS TO TRUSTED LOCATIONS</b><br>"
             "<sup>How many dark-store observations survive each geographic quality gate?</sup>",
        x=0.5,
        xanchor="center",
        font=dict(size=22)
    ),

    paper_bgcolor="white",

    height=600,

    margin=dict(
        t=120,
        l=150,
        r=80,
        b=60
    )
)

fig.show()